## 1. Install Libraries

In [1]:
!pip install -q faiss-gpu

!pip install -q sentence-transformers

In [2]:
import faiss

print("FAISS version:", faiss.__version__)

print("GPU supported:", hasattr(faiss, "StandardGpuResources"))

FAISS version: 1.14.3
GPU supported: True


In [3]:
import psutil

print(f"RAM: {psutil.virtual_memory().total / (1024**3):.2f} GB")

RAM: 12.67 GB


## 2. Import Libraries

In [4]:
import os
import gc
import time
import warnings

import numpy as np
import pandas as pd

import torch
import faiss

from tqdm.auto import tqdm

from sentence_transformers import SentenceTransformer

warnings.filterwarnings("ignore")

## 3. Project Paths

In [5]:
PROJECT_PATH = "/content/drive/MyDrive/FinSight_AI"

DATA_PATH = os.path.join(
    PROJECT_PATH,
    "data"
)

EMBEDDING_PATH = os.path.join(
    DATA_PATH,
    "embeddings"
)

PROCESSED_PATH = os.path.join(
    DATA_PATH,
    "processed"
)

EMBEDDING_BATCH_PATH = os.path.join(
    EMBEDDING_PATH,
    "embedding_batches"
)

REPORT_PATH = os.path.join(
    PROJECT_PATH,
    "reports"
)

## 4. Load Manifest

In [6]:
manifest_df = pd.read_parquet(

    os.path.join(

        EMBEDDING_PATH,

        "embedding_manifest.parquet"

    )

)

manifest_df.head()

,batch_number,start_row,end_row,rows,embedding_dimension,embedding_file,metadata_file
0,1,0,4999,5000,768,embedding_batch_0001.npy,metadata_batch_0001.parquet
1,2,5000,9999,5000,768,embedding_batch_0002.npy,metadata_batch_0002.parquet
2,3,10000,14999,5000,768,embedding_batch_0003.npy,metadata_batch_0003.parquet
3,4,15000,19999,5000,768,embedding_batch_0004.npy,metadata_batch_0004.parquet
4,5,20000,24999,5000,768,embedding_batch_0005.npy,metadata_batch_0005.parquet


## 5. Load Embedding Index

In [7]:
# embedding_index = pd.read_parquet(

#     os.path.join(

#         EMBEDDING_PATH,

#         "embedding_index.parquet"

#     )

# )

# embedding_index.head()

## 6. Verify

In [8]:
print(manifest_df.shape)

# print(embedding_index.shape)

(644, 7)


## 7. GPU Check

In [9]:
print(

    "CUDA Available:",

    torch.cuda.is_available()

)

if torch.cuda.is_available():

    print(

        "GPU:",

        torch.cuda.get_device_name(0)

    )

CUDA Available: True
GPU: Tesla T4


## 8. Load SentenceTransformer

In [10]:
model = SentenceTransformer(

    "sentence-transformers/all-mpnet-base-v2",

    device="cuda"

)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## 9. FAISS Information

In [11]:
EMBEDDING_DIM = 768

TOTAL_BATCHES = len(manifest_df)

print(

    "Embedding Dimension:",

    EMBEDDING_DIM

)

print(

    "Total Batches:",

    TOTAL_BATCHES

)

Embedding Dimension: 768
Total Batches: 644


## 10. Create FAISS Index

In [12]:
index = faiss.IndexFlatIP(

    EMBEDDING_DIM

)

print(index)

<faiss.swigfaiss.IndexFlatIP; proxy of <Swig Object of type 'faiss::IndexFlatIP *' at 0x7a852ce0e070> >


## 11. Create Save Paths

In [13]:
FAISS_INDEX_PATH = os.path.join(

    EMBEDDING_PATH,

    "faiss_index.bin"

)

CHECKPOINT_PATH = os.path.join(

    EMBEDDING_PATH,

    "faiss_progress.json"

)

## 12. Checkpoint Function

In [14]:
import json
from datetime import datetime

def save_checkpoint(

    completed_batches,

    total_vectors

):

    checkpoint = {

        "completed_batches": completed_batches,

        "total_vectors": total_vectors,

        "last_updated": datetime.now().strftime(

            "%Y-%m-%d %H:%M:%S"

        )

    }

    with open(

        CHECKPOINT_PATH,

        "w"

    ) as f:

        json.dump(

            checkpoint,

            f,

            indent=4

        )

## 13. Verify Empty Index

In [15]:
print(

    "Vectors currently in index :",

    index.ntotal

)

Vectors currently in index : 0


## 14. Test One Batch

In [16]:
# sample_embeddings = np.load(

#     os.path.join(

#         EMBEDDING_BATCH_PATH,

#         manifest_df.loc[0,"embedding_file"]

#     )

# )

# print(sample_embeddings.shape)

## 15. Add Test Batch

In [17]:
manifest_df = manifest_df[
    manifest_df["embedding_file"] != "test_embedding.npy"
].reset_index(drop=True)

manifest_df.to_parquet(
    os.path.join(
        EMBEDDING_PATH,
        "embedding_manifest.parquet"
    ),
    index=False
)

TOTAL_BATCHES = len(manifest_df)

print(TOTAL_BATCHES)

644


## 16. Reset Test Index

In [18]:
NLIST = 4096

M = 32          # number of sub-vectors

NBITS = 8       # bits per code

quantizer = faiss.IndexFlatIP(EMBEDDING_DIM)

index = faiss.IndexIVFPQ(
    quantizer,
    EMBEDDING_DIM,
    NLIST,
    M,
    NBITS,
    faiss.METRIC_INNER_PRODUCT
)

print(index)

<faiss.swigfaiss.IndexIVFPQ; proxy of <Swig Object of type 'faiss::IndexIVFPQ *' at 0x7a852cdfcc70> >


In [19]:
train_batches = []

for i in range(20):      # first 100k vectors

    file = os.path.join(
        EMBEDDING_BATCH_PATH,
        manifest_df.loc[i, "embedding_file"]
    )

    train_batches.append(
        np.load(file).astype(np.float32)
    )

train_vectors = np.vstack(train_batches)

print(train_vectors.shape)

index.train(train_vectors)

index.nprobe = 256

print("Index trained:", index.is_trained)

del train_vectors
del train_batches

gc.collect()

(100000, 768)
Index trained: True


375

## 17. Production FAISS Build Loop

In [20]:
# manifest_df.to_parquet(

#     os.path.join(

#         EMBEDDING_PATH,

#         "embedding_manifest.parquet"

#     ),

#     index=False

# )

# print("Manifest corrected and saved.")

In [21]:
# TOTAL_BATCHES = len(manifest_df)

# print(TOTAL_BATCHES)

In [22]:
overall_start = time.time()

for batch_number in tqdm(

    range(TOTAL_BATCHES),

    desc="Building FAISS Index"

):

    embedding_file = os.path.join(

        EMBEDDING_BATCH_PATH,

        manifest_df.loc[
            batch_number,
            "embedding_file"
        ]

    )

    batch_start = time.time()

    embeddings = np.load(
        embedding_file,
        mmap_mode="r"
    )

    embeddings = np.asarray(
        embeddings,
        dtype=np.float32
    )

    index.add(
        embeddings
    )

    save_checkpoint(
        batch_number + 1,
        index.ntotal
    )

    elapsed = time.time() - overall_start

    avg_batch = elapsed / (batch_number + 1)

    remaining = TOTAL_BATCHES - (batch_number + 1)

    eta = remaining * avg_batch / 60

    if (batch_number + 1) % 25 == 0 or batch_number == TOTAL_BATCHES - 1:

        print("=" * 60)

        print(f"Batch : {batch_number+1}/{TOTAL_BATCHES}")

        print(f"Indexed : {index.ntotal:,}")

        print(f"ETA : {eta:.2f} minutes")

    del embeddings

    gc.collect()

    torch.cuda.empty_cache()

Building FAISS Index:   0%|          | 0/644 [00:00<?, ?it/s]

Batch : 25/644
Indexed : 125,000
ETA : 10.86 minutes
Batch : 50/644
Indexed : 250,000
ETA : 9.24 minutes
Batch : 75/644
Indexed : 375,000
ETA : 8.60 minutes
Batch : 100/644
Indexed : 500,000
ETA : 8.06 minutes
Batch : 125/644
Indexed : 625,000
ETA : 7.68 minutes
Batch : 150/644
Indexed : 750,000
ETA : 7.25 minutes
Batch : 175/644
Indexed : 875,000
ETA : 6.87 minutes
Batch : 200/644
Indexed : 1,000,000
ETA : 6.49 minutes
Batch : 225/644
Indexed : 1,125,000
ETA : 6.06 minutes
Batch : 250/644
Indexed : 1,250,000
ETA : 5.72 minutes
Batch : 275/644
Indexed : 1,375,000
ETA : 5.32 minutes
Batch : 300/644
Indexed : 1,500,000
ETA : 4.97 minutes
Batch : 325/644
Indexed : 1,625,000
ETA : 4.59 minutes
Batch : 350/644
Indexed : 1,750,000
ETA : 4.24 minutes
Batch : 375/644
Indexed : 1,875,000
ETA : 3.86 minutes
Batch : 400/644
Indexed : 2,000,000
ETA : 3.50 minutes
Batch : 425/644
Indexed : 2,125,000
ETA : 3.14 minutes
Batch : 450/644
Indexed : 2,250,000
ETA : 2.80 minutes
Batch : 475/644
Indexed : 

In [23]:
manifest_df.tail(10)

,batch_number,start_row,end_row,rows,embedding_dimension,embedding_file,metadata_file
634,635,3170000,3174999,5000,768,embedding_batch_0635.npy,metadata_batch_0635.parquet
635,636,3175000,3179999,5000,768,embedding_batch_0636.npy,metadata_batch_0636.parquet
636,637,3180000,3184999,5000,768,embedding_batch_0637.npy,metadata_batch_0637.parquet
637,638,3185000,3189999,5000,768,embedding_batch_0638.npy,metadata_batch_0638.parquet
638,639,3190000,3194999,5000,768,embedding_batch_0639.npy,metadata_batch_0639.parquet
639,640,3195000,3199999,5000,768,embedding_batch_0640.npy,metadata_batch_0640.parquet
640,641,3200000,3204999,5000,768,embedding_batch_0641.npy,metadata_batch_0641.parquet
641,642,3205000,3209999,5000,768,embedding_batch_0642.npy,metadata_batch_0642.parquet
642,643,3210000,3214999,5000,768,embedding_batch_0643.npy,metadata_batch_0643.parquet
643,644,3215000,3215295,296,768,embedding_batch_0644.npy,metadata_batch_0644.parquet


## 18. Verify Final Index

In [24]:
print(

    "Total Vectors Stored:",

    index.ntotal

)

Total Vectors Stored: 3215296


## 19. Save FAISS Index

In [25]:
# Save and Free RAM

print("\nSaving FAISS index...")

faiss.write_index(
    index,
    FAISS_INDEX_PATH
)

config = {

    "dimension": EMBEDDING_DIM,

    "nlist": NLIST,

    "nprobe": index.nprobe,

    "metric": "Inner Product"

}

with open(

    os.path.join(

        EMBEDDING_PATH,

        "faiss_config.json"

    ),

    "w"

) as f:

    json.dump(

        config,

        f,

        indent=4

    )

print("FAISS index saved successfully.")

# Free memory
del index

import gc

gc.collect()

print("RAM cleaned.")
print(

    "FAISS Index Saved Successfully"

)


Saving FAISS index...
FAISS index saved successfully.
RAM cleaned.
FAISS Index Saved Successfully


## 20. Reload Index

In [26]:
loaded_index = faiss.read_index(

    FAISS_INDEX_PATH

)

loaded_index.nprobe = 256

print(

    loaded_index.ntotal

)

3215296


## 21. Summary

In [27]:
summary = pd.DataFrame({

    "Metric":[

        "Embedding Dimension",

        "Total Embeddings",

        "Total Batches",

        "FAISS Index Type"

    ],

    "Value":[

        EMBEDDING_DIM,

        loaded_index.ntotal,

        TOTAL_BATCHES,

        "IndexIVFPQ"

    ]

})

summary

,Metric,Value
0,Embedding Dimension,768
1,Total Embeddings,3215296
2,Total Batches,644
3,FAISS Index Type,IndexIVFPQ


In [28]:
embedding_index = pd.read_parquet(

    os.path.join(

        EMBEDDING_PATH,

        "embedding_index.parquet"

    )

)

## 22. Encode Query

In [29]:
def encode_query(query):

    embedding = model.encode(

        query,

        normalize_embeddings=True,

        convert_to_numpy=True

    )

    return embedding.astype(np.float32).reshape(1, -1)

## 23. Test Query Encoding

In [30]:
query = "Apple reports strong quarterly earnings"

query_embedding = encode_query(query)

print(query_embedding.shape)

(1, 768)


## 24. FAISS Search Function

In [31]:
def semantic_search(

    query,

    top_k=5

):

    query_embedding = encode_query(query)

    scores, indices = loaded_index.search(

        query_embedding,

        top_k

    )

    results = embedding_index.iloc[
        indices[0]
    ].copy()

    results["similarity"] = scores[0]

    return results

## 25. First Search

In [32]:
semantic_search(

    "Apple reports strong quarterly earnings"

)

,embedding_id,news_id,ticker,published_date,source,batch_number,row_in_batch,similarity
1679296,1679296,1679302,CHL,2014-01-24 00:00:00+00:00,partner,336,4296,0.636950
2246535,2246535,2246547,IP,2015-01-28 00:00:00+00:00,partner,450,1535,0.635706
2243621,2243621,2243633,INVN,2015-04-28 00:00:00+00:00,partner,449,3621,0.635289
1752881,1752881,1752887,CRUS,2015-04-28 00:00:00+00:00,partner,351,2881,0.635289
1404109,1404109,1404113,ABTL,2015-04-28 00:00:00+00:00,partner,281,4109,0.635289


## 26. Recover Original Headlines

In [33]:
def get_headlines(news_ids):

    headlines = pd.read_parquet(

        os.path.join(

            PROCESSED_PATH,

            "financial_news_clean.parquet"

        ),

        columns=[

            "news_id",

            "headline"

        ]

    )

    headlines = headlines[

        headlines["news_id"].isin(news_ids)

    ]

    headlines = (

        headlines

        .set_index("news_id")

        .loc[news_ids]

        .reset_index()

    )

    return headlines

## 27. Merge Headlines

In [34]:
def search_news(

    query,

    top_k=5

):

    results = semantic_search(

        query,

        top_k

    )

    headlines = get_headlines(

        results["news_id"].tolist()

    )

    results = results.merge(

        headlines,

        on="news_id",

        how="left"

    )

    return results[

        [

            "headline",

            "ticker",

            "published_date",

            "similarity"

        ]

    ]

## 28. Pretty Search Function

In [35]:
search_news(

    "Apple reports strong quarterly earnings",

    top_k=5

)

,headline,ticker,published_date,similarity
0,"Apple Q1 Earnings Seen Boosted By New iPhones,...",CHL,2014-01-24 00:00:00+00:00,0.636950
1,Apple Earnings Boost Overall Q4 Growth - Ahead...,IP,2015-01-28 00:00:00+00:00,0.635706
2,"Apple (AAPL) Tops Q2 Earnings on Solid iPhone,...",INVN,2015-04-28 00:00:00+00:00,0.635289
3,"Apple (AAPL) Tops Q2 Earnings on Solid iPhone,...",CRUS,2015-04-28 00:00:00+00:00,0.635289
4,"Apple (AAPL) Tops Q2 Earnings on Solid iPhone,...",ABTL,2015-04-28 00:00:00+00:00,0.635289


## 29. Test Searches

In [36]:
queries = [

    "Apple earnings",

    "Tesla stock falls",

    "Dividend increase",

    "Oil prices rise",

    "Federal Reserve interest rates"

]

for q in queries:

    print("=" * 80)

    print(q)

    print()

    display(

        search_news(

            q,

            top_k=5

        )

    )

Apple earnings



,headline,ticker,published_date,similarity
0,Apple Earnings: Taking A Long-Term View,TSM,2016-01-27 00:00:00+00:00,0.756147
1,Apple Shares Have Had Quite A Run; What's Next...,NFLX,2020-01-27 00:00:00+00:00,0.740677
2,Apple Earnings: Outstanding Achievement in the...,NVDA,2012-01-25 00:00:00+00:00,0.738086
3,Apple Earnings: Outstanding Achievement in the...,NUAN,2012-01-25 00:00:00+00:00,0.738086
4,Apple Earnings: Outstanding Achievement in the...,MMI,2012-01-25 00:00:00+00:00,0.738086


Tesla stock falls



,headline,ticker,published_date,similarity
0,Tesla Shares Fall ~$108 Over Last Few Mins,TSLA,2020-02-04 00:00:00+00:00,0.553935
1,"Tesla Sets Record For Deliveries, But Stock Pr...",TSLA,2019-10-04 00:00:00+00:00,0.546333
2,"Tesla Sets Record For Deliveries, But Stock Pr...",TM,2019-10-04 00:00:00+00:00,0.546333
3,"Tesla Sets Record For Deliveries, But Stock Pr...",HMC,2019-10-04 00:00:00+00:00,0.546333
4,"Tesla Sets Record For Deliveries, But Stock Pr...",F,2019-10-04 00:00:00+00:00,0.546333


Dividend increase



,headline,ticker,published_date,similarity
0,This Comapny Is Spicing Up Its Dividend With A...,OGE,2012-11-30 00:00:00+00:00,0.724064
1,This Comapny Is Spicing Up Its Dividend With A...,MKC,2012-11-30 00:00:00+00:00,0.724064
2,This Comapny Is Spicing Up Its Dividend With A...,FISI,2012-11-30 00:00:00+00:00,0.724064
3,This Comapny Is Spicing Up Its Dividend With A...,CFFI,2012-11-30 00:00:00+00:00,0.724064
4,This Comapny Is Spicing Up Its Dividend With A...,BKU,2012-11-30 00:00:00+00:00,0.724064


Oil prices rise



,headline,ticker,published_date,similarity
0,Oil Prices To Rise,USL,2019-09-30 00:00:00+00:00,0.681406
1,Oil Prices To Rise,SCO,2019-09-30 00:00:00+00:00,0.681406
2,"Oil Prices Rise (SCO, DBO)",SCO,2010-05-12 00:00:00+00:00,0.642847
3,"Oil Prices Rise (SCO, DBO)",DBO,2010-05-12 00:00:00+00:00,0.642847
4,Oil Prices On The Rise,BNO,2014-03-04 00:00:00+00:00,0.639597


Federal Reserve interest rates



,headline,ticker,published_date,similarity
0,Financials - Interest Rates And Central Banks,TD,2016-03-24 00:00:00+00:00,0.620067
1,Financials - Interest Rates And Central Banks,CM,2016-03-24 00:00:00+00:00,0.620067
2,Financials - Interest Rates And Central Banks,BNS,2016-03-24 00:00:00+00:00,0.620067
3,Financials - Interest Rates And Central Banks,BMO,2016-03-24 00:00:00+00:00,0.620067
4,Fed Funds Rate At 7.5%?,ETP,2018-08-29 00:00:00+00:00,0.609037


## 30. Similarity Threshold Search

In [37]:
def search_above_threshold(

    query,

    threshold=0.75,

    top_k=20

):

    results = semantic_search(

        query,

        top_k

    )

    headlines = get_headlines(

        results["news_id"].tolist()

    )

    results = results.merge(

        headlines,

        on="news_id",

        how="left"

    )

    results = results[

        results["similarity"] >= threshold

    ]

    return results[
        [
            "headline",
            "ticker",
            "published_date",
            "similarity"
        ]
    ]

## 31. Test Threshold Search

In [38]:
search_above_threshold(

    "Apple earnings",

    threshold=0.80,

    top_k=20

)

,headline,ticker,published_date,similarity


## 32. Final Statistics

In [39]:
summary = pd.DataFrame({

    "Metric":[

        "Indexed Headlines",

        "Embedding Dimension",

        "FAISS Index",

        "Similarity Metric"

    ],

    "Value":[

        loaded_index.ntotal,

        768,

        "IndexIVFPQ",

        "Cosine Similarity"

    ]

})

summary

,Metric,Value
0,Indexed Headlines,3215296
1,Embedding Dimension,768
2,FAISS Index,IndexIVFPQ
3,Similarity Metric,Cosine Similarity
